# 01 — Dataset Exploration and Sampling Audit

**Study:** Understanding the Structure and Reuse of AI Agent Skills in Open-Source Software Projects  
**Dataset:** GitSkills sample (July 2026 collection)

This notebook examines the composition of the sample before the research-question analyses. It records the number of skill-file occurrences, distinct contents and repositories, checks metadata coverage, and describes how unevenly occurrences are distributed across repositories.

The sample contains the 13,000 lowest Git blob hashes from the collected dataset, together with all recorded occurrences of those contents. Repositories were not sampled independently. This selection rule comes from the GitSkills sample documentation; the database alone does not verify the ordering against the full collection.

The analyses below describe the supplied sample rather than all GitHub repositories. The notebook runs independently of the other analyses.

## 1. Database connection

The SQLite database is expected at `data/raw/agent_skills_sample.db`, relative to the project root. The connection is opened in read-only mode. The initial query checks that the main tables are available before further analysis.

The notebook uses Python's `sqlite3` module together with pandas, NumPy and Matplotlib.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the project root
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "data" / "raw" / "agent_skills_sample.db"

assert DB_PATH.is_file(), f"Database not found: {DB_PATH}"

conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)

tables = [
    "mining_runs",
    "repos",
    "artifact_siblings",
    "artifacts"
]

table_counts = []

for table in tables:
    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS count FROM {table}",
        conn
    ).iloc[0]["count"]

    table_counts.append({
        "table": table,
        "records": int(count)
    })

counts_df = pd.DataFrame(table_counts)

display(counts_df)

,table,records
0,mining_runs,7
1,repos,11841
2,artifact_siblings,47829
3,artifacts,29786


## 2. Dataset coverage

The first query counts recorded skill-file occurrences, distinct Git blob hashes, repositories, fetched content records, and available metadata. The `artifacts` table contains one row per recorded occurrence, so the number of rows exceeds the number of distinct file contents.

The reference sample contains 29,786 occurrences, 13,000 distinct hashes and 11,841 repositories. Content is generally fetched for primary representatives rather than repeated for every occurrence. Consequently, counts of fetched rows should not be read as counts of distinct contents with available text.

In [2]:
# Dataset coverage and completeness

coverage_query = """
SELECT
    COUNT(*) AS total_artifacts,
    COUNT(DISTINCT repo_full_name) AS unique_repositories,
    COUNT(DISTINCT repo_full_name || '::' || path) AS unique_artifact_paths,
    SUM(CASE WHEN content_fetched = 1 THEN 1 ELSE 0 END) AS content_fetched,
    SUM(CASE WHEN content IS NOT NULL THEN 1 ELSE 0 END) AS content_available,
    SUM(CASE WHEN file_sha IS NOT NULL THEN 1 ELSE 0 END) AS file_sha_available,
    SUM(CASE WHEN frontmatter_valid = 1 THEN 1 ELSE 0 END) AS valid_frontmatter,
    SUM(CASE WHEN composition_fetched = 1 THEN 1 ELSE 0 END) AS composition_fetched
FROM artifacts
"""

coverage_df = pd.read_sql_query(coverage_query, conn)

display(coverage_df.T.rename(columns={0: "Value"}))

,Value
total_artifacts,29786
unique_repositories,11841
unique_artifact_paths,29786
content_fetched,13016
content_available,13016
file_sha_available,29786
valid_frontmatter,11305
composition_fetched,13006


### 2.1 Missing values

The following summary reports missingness for variables used later in the study. Some null values reflect the dataset's storage design: duplicated occurrences may omit text because a primary record already stores that content. Likewise, missing composition metadata is different from a recorded count of zero supporting files.

Front-matter counts in this occurrence-level audit should not be compared directly with RQ1 prevalence. RQ1 uses one primary representative per distinct hash, which has a different denominator.

In [3]:
# Missing values in research-relevant attributes

important_columns = [
    "repo_full_name",
    "path",
    "file_sha",
    "content",
    "name",
    "description",
    "body_chars",
    "sibling_count",
    "has_scripts",
    "has_references",
    "frontmatter_valid"
]

artifacts_df = pd.read_sql_query(
    f"SELECT {', '.join(important_columns)} FROM artifacts",
    conn
)

missing_df = pd.DataFrame({
    "Missing": artifacts_df.isna().sum(),
    "Missing (%)": (
        artifacts_df.isna().mean() * 100
    ).round(2)
})

display(missing_df)

,Missing,Missing (%)
repo_full_name,0,0.00
path,0,0.00
file_sha,0,0.00
content,16770,56.30
name,18433,61.88
description,18332,61.55
body_chars,16770,56.30
sibling_count,16780,56.34
has_scripts,16780,56.34
has_references,16780,56.34


## 3. Content availability across repositories

For each repository, we calculate the proportion of its recorded occurrences with fetched text. This describes how content records are distributed across repository rows; it is not a measure of repository sampling probability or completeness.

Because text is deduplicated by hash, a repository can have a low fetched-text rate even when the contents of its files are available through primary records elsewhere.

In [4]:
# Content availability by repository

availability_query = """
SELECT
    a.repo_full_name,
    COUNT(*) AS total_skills,
    SUM(CASE WHEN a.content_fetched = 1 THEN 1 ELSE 0 END)
        AS fetched_skills,
    r.stars,
    r.language,
    r.is_fork
FROM artifacts a
LEFT JOIN repos r
    ON a.repo_full_name = r.full_name
GROUP BY a.repo_full_name
"""

availability_df = pd.read_sql_query(
    availability_query,
    conn
)

availability_df["fetch_rate"] = (
    availability_df["fetched_skills"]
    / availability_df["total_skills"]
)

print("Repositories:", len(availability_df))

print("\nFetch rate summary:")
display(availability_df["fetch_rate"].describe())

print("\nRepositories by content availability:")

print(
    "All content fetched:",
    (availability_df["fetch_rate"] == 1).sum()
)

print(
    "Partially fetched:",
    (
        (availability_df["fetch_rate"] > 0)
        & (availability_df["fetch_rate"] < 1)
    ).sum()
)

print(
    "No content fetched:",
    (availability_df["fetch_rate"] == 0).sum()
)

Repositories: 11841

Fetch rate summary:


count    11841.000000
mean         0.634964
std          0.465124
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: fetch_rate, dtype: float64


Repositories by content availability:
All content fetched: 7137
Partially fetched: 831
No content fetched: 3873


### 3.1 Content availability by star count

We group repositories by their recorded GitHub star counts and compare the mean fetched-text rate within each group. Repositories without star metadata are excluded from this comparison.

Any differences are descriptive. Primary-representative selection affects the fetched-text rate, so the comparison cannot establish that repository popularity influences content retrieval.

In [5]:
# Compare content availability across star-count groups

availability_df["star_group"] = pd.cut(
    availability_df["stars"],
    bins=[-1, 0, 10, 100, 1000, float("inf")],
    labels=[
        "0 stars",
        "1–10 stars",
        "11–100 stars",
        "101–1000 stars",
        ">1000 stars"
    ]
)

star_summary = (
    availability_df
    .groupby("star_group", observed=True)
    .agg(
        repositories=("repo_full_name", "count"),
        mean_fetch_rate=("fetch_rate", "mean"),
        median_fetch_rate=("fetch_rate", "median")
    )
    .reset_index()
)

star_summary["mean_fetch_rate"] = (
    star_summary["mean_fetch_rate"] * 100
).round(2)

star_summary["median_fetch_rate"] = (
    star_summary["median_fetch_rate"] * 100
).round(2)

display(star_summary)

,star_group,repositories,mean_fetch_rate,median_fetch_rate
0,0 stars,6424,58.83,100.0
1,1–10 stars,3723,69.85,100.0
2,11–100 stars,1041,72.86,100.0
3,101–1000 stars,407,66.78,100.0
4,>1000 stars,209,49.96,50.0


## 4. Collection records and artifact locations

Collection-run metadata provides context for the July 2026 snapshot. We also summarize the recorded artifact-location categories (`canonical`, `skills-dir`, and `other`). These labels describe discovery locations; they do not establish whether individual files comply with the Agent Skills specification.

The source collection relied on GitHub search and therefore inherits its indexing and default-branch coverage limitations.

In [6]:
# Inspect mining runs and discovery metadata

mining_runs_df = pd.read_sql_query(
    """
    SELECT
        run_id,
        artifact_type,
        query,
        started_at,
        finished_at,
        discovered,
        note
    FROM mining_runs
    ORDER BY run_id
    """,
    conn
)

display(mining_runs_df)

print("\nArtifact location classes:")

location_df = pd.read_sql_query(
    """
    SELECT
        location_class,
        COUNT(*) AS artifacts,
        COUNT(DISTINCT repo_full_name) AS repositories
    FROM artifacts
    GROUP BY location_class
    ORDER BY artifacts DESC
    """,
    conn
)

display(location_df)

,run_id,artifact_type,query,started_at,finished_at,discovered,note
0,1,agent-skills,filename:SKILL.md,2026-07-09T17:26:45.768703+00:00,2026-07-09T18:04:34.751737+00:00,11796,NaN
1,2,agent-skills,filename:SKILL.md,2026-07-09T18:07:39.150068+00:00,2026-07-12T06:54:51.308396+00:00,1328473,NaN
2,3,agent-skills,filename:SKILL.md,2026-07-12T06:59:13.601456+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
3,4,agent-skills,filename:SKILL.md,2026-07-13T12:52:52.309967+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
4,5,agent-skills,filename:SKILL.md,2026-07-14T18:04:23.050325+00:00,2026-07-20T05:23:31.638481+00:00,1657948,NaN
5,6,agent-skills,filename:SKILL.md,2026-07-20T05:28:44.667482+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
6,7,agent-skills,filename:SKILL.md,2026-07-20T07:38:28.581854+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...



Artifact location classes:


,location_class,artifacts,repositories
0,skills-dir,14882,7671
1,other,12539,2806
2,canonical,2365,2020


### 4.1 Repository metadata coverage

The next query checks the completeness of repository attributes used in the analysis. In particular, an unknown `is_fork` value is treated as missing information, not as evidence for either fork status.

In [7]:
# Repository metadata completeness

repo_metadata_df = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_repositories,

        SUM(CASE WHEN metadata_fetched = 1
            THEN 1 ELSE 0 END) AS metadata_fetched,

        SUM(CASE WHEN stars IS NULL
            THEN 1 ELSE 0 END) AS missing_stars,

        SUM(CASE WHEN language IS NULL
            THEN 1 ELSE 0 END) AS missing_language,

        SUM(CASE WHEN license IS NULL
            THEN 1 ELSE 0 END) AS missing_license,

        SUM(CASE WHEN is_fork IS NULL
            THEN 1 ELSE 0 END) AS missing_fork_status
    FROM repos
    """,
    conn
)

display(repo_metadata_df.T.rename(columns={0: "Value"}))

,Value
total_repositories,11841
metadata_fetched,11841
missing_stars,37
missing_language,37
missing_license,37
missing_fork_status,37


## 5. Concentration of occurrences across repositories

We count occurrences per repository and measure the contribution of the largest 1% of repositories, rounding the number of repositories upward. This helps identify whether a small number of large collections dominate the sample.

The result motivates the repository-size sensitivity analysis in Notebook 05. It should not be generalized to the population of GitHub repositories, because selection was based on content hashes.

In [8]:
# Distribution of artifacts per repository

repo_distribution_query = """
SELECT
    repo_full_name,
    COUNT(*) AS artifact_count,
    COUNT(DISTINCT file_sha) AS unique_hashes
FROM artifacts
GROUP BY repo_full_name
"""

repo_distribution_df = pd.read_sql_query(
    repo_distribution_query,
    conn
)

print("Repositories:", len(repo_distribution_df))

display(
    repo_distribution_df["artifact_count"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nTop 10 repositories by artifact count:")

display(
    repo_distribution_df
    .sort_values("artifact_count", ascending=False)
    .head(10)
)

# Contribution of the top 1% of repositories

top_n = max(1, int(np.ceil(len(repo_distribution_df) * 0.01)))

top_share = (
    repo_distribution_df
    .nlargest(top_n, "artifact_count")["artifact_count"]
    .sum()
    / repo_distribution_df["artifact_count"].sum()
    * 100
)

print(
    f"\nTop 1% of repositories ({top_n}) "
    f"account for {top_share:.2f}% of artifacts."
)

Repositories: 11841


count    11841.000000
mean         2.515497
std         34.967723
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
90%          2.000000
95%          4.000000
99%         17.600000
max       2952.000000
Name: artifact_count, dtype: float64


Top 10 repositories by artifact count:


,repo_full_name,artifact_count,unique_hashes
8127,majiayu000/claude-skill-registry-data,2952,1300
5537,diegosouzapw/awesome-omni-skill,1886,1071
2249,MikeCheng1208/BattleTree,813,813
927,David-Li0406/meta-skill-evloving,728,269
8126,majiayu000/claude-skill-registry,593,268
2415,NeuralBlitz/ncx,422,287
6292,gabrielmoreira/agent-skills-mirror,416,291
2414,NeuralBlitz/buggy,204,204
10880,tools-only/X-Skills,164,164
5711,dvcrn/openclaw-skills-marketplace,155,145



Top 1% of repositories (119) account for 42.32% of artifacts.


## 6. Deduplication bookkeeping

Each distinct hash should have one designated primary representative. We check the number of primary rows and look for hashes with no primary or multiple primaries.

These checks test internal consistency. They cannot confirm that the selected hashes were the lowest 13,000 in the full GitSkills collection, which would require the full dataset.

In [9]:

# Audit sample selection indicators

selection_audit = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_artifacts,
        COUNT(DISTINCT file_sha) AS unique_hashes,
        SUM(CASE WHEN dedup_primary = 1 THEN 1 ELSE 0 END)
            AS primary_representatives,
        SUM(CASE WHEN dedup_primary = 0 THEN 1 ELSE 0 END)
            AS non_primary_occurrences,
        SUM(CASE WHEN content_fetched = 1 THEN 1 ELSE 0 END)
            AS fetched_contents,
        COUNT(DISTINCT CASE
            WHEN content_fetched = 1 THEN file_sha
        END) AS hashes_with_fetched_content
    FROM artifacts
    """,
    conn
)

display(selection_audit)

# Count primary representatives per content hash.
primary_per_hash = pd.read_sql_query(
    """
    SELECT
        file_sha,
        COUNT(*) AS occurrences,
        SUM(CASE WHEN dedup_primary = 1 THEN 1 ELSE 0 END)
            AS primary_count
    FROM artifacts
    GROUP BY file_sha
    """,
    conn
)

print("Hashes with exactly one primary representative:",
      (primary_per_hash["primary_count"] == 1).sum())

print("Hashes with no primary representative:",
      (primary_per_hash["primary_count"] == 0).sum())

print("Hashes with multiple primary representatives:",
      (primary_per_hash["primary_count"] > 1).sum())


,total_artifacts,unique_hashes,primary_representatives,non_primary_occurrences,fetched_contents,hashes_with_fetched_content
0,29786,13000,13000,16786,13016,13000


Hashes with exactly one primary representative: 13000
Hashes with no primary representative: 0
Hashes with multiple primary representatives: 0


## 7. Checks against the reference sample

The assertions below compare key counts with the sample used for this study. They are intended to catch a different database version, an unexpected change in preprocessing, or a broken assumption. They are not general validity tests for arbitrary GitSkills releases.

In [10]:
assert int(coverage_df.loc[0, "total_artifacts"]) == 29786
assert int(coverage_df.loc[0, "unique_repositories"]) == 11841
assert int(selection_audit.loc[0, "unique_hashes"]) == 13000
assert int(selection_audit.loc[0, "primary_representatives"]) == 13000
assert (primary_per_hash["primary_count"] == 1).all()
assert top_n == 119
print("Dataset exploration and sampling bookkeeping checks passed.")

Dataset exploration and sampling bookkeeping checks passed.


## 8. Summary and limitations

The sample comprises 29,786 occurrences of 13,000 selected contents across 11,841 repositories. Its hash-based selection preserves recorded occurrences of selected contents but does not yield a random sample of repositories.

Text stored only on primary records should not be mistaken for missing skill content. The RQ1 analysis therefore uses primary representatives, while the RQ2 analysis includes all recorded occurrences. The uneven repository-size distribution also warrants a separate sensitivity analysis.

Finally, the July 2026 GitHub-search snapshot has coverage limits, including search indexing and filename-based discovery. These results describe the sampled records and cannot establish how skills spread between repositories.

Notebook 02 examines the integrity of content hashes, deduplication records, and supporting-file metadata.